In [1]:
import pandas as pd
df= pd.read_csv("Fake.csv")

In [2]:
df.shape

(23481, 4)

In [3]:
df.head(4)

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"


In [4]:
df['subject'].value_counts()

subject
News               9050
politics           6841
left-news          4459
Government News    1570
US_News             783
Middle-east         778
Name: count, dtype: int64

In [5]:
df= df.drop(columns=["title","date"])

In [6]:
df.sample(3)

,text,subject
7660,Donald Trump sat down for an interview with on...,News
7461,"As we all remember last year, Pope Francis mad...",News
8951,Anyone who watched Barack Obama break into tea...,News


In [7]:
df["label_num"] = (
    df["subject"]
    .map({
        "News": 0,
        "politics": 1,
        "left-news": 2,
        "Government News": 3,
        "US_News": 4,
        "Middle_east": 5
    })
    .astype("Int64")
)

In [8]:
df.head(5)

,text,subject,label_num
0,Donald Trump just couldn t wish all Americans ...,News,0
1,House Intelligence Committee Chairman Devin Nu...,News,0
2,"On Friday, it was revealed that former Milwauk...",News,0
3,"On Christmas day, Donald Trump announced that ...",News,0
4,Pope Francis used his annual Christmas Day mes...,News,0


In [9]:
df = df.dropna(subset=["label_num"])

In [10]:
from sklearn.model_selection import train_test_split


#Do the 'train-test' splitting with test size of 20% with random state of 2022 and stratify sampling too
X_train, X_test, y_train, y_test = train_test_split(
    df.text, 
    df.label_num, 
    test_size=0.2, # 20% samples will go to test dataset
    random_state=2022,
    stratify=df.label_num
)


In [11]:
print("Shape of X_train: ", X_train.shape)
print("Shape of X_test: ", X_test.shape)

Shape of X_train:  (18162,)
Shape of X_test:  (4541,)


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from  sklearn.neighbors import KNeighborsClassifier

#1. create a pipeline object
clf = Pipeline([
    ('vectorizer_trigrams', CountVectorizer(ngram_range = (1, 1))),                   #using the ngram_range parameter 
     ('KNN', (KNeighborsClassifier(n_neighbors=10, metric = 'euclidean')))           #using the KNN classifier with 10 neighbors and euclidean distance      
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.70      0.84      0.77      1810
         1.0       0.43      0.62      0.51      1368
         2.0       0.21      0.07      0.11       892
         3.0       0.12      0.01      0.02       314
         4.0       0.79      0.43      0.56       157

    accuracy                           0.55      4541
   macro avg       0.45      0.39      0.39      4541
weighted avg       0.49      0.55      0.50      4541



In [13]:
from sklearn.naive_bayes import MultinomialNB


#1. create a pipeline object
clf = Pipeline([
    ('vectorizer_trigrams', CountVectorizer(ngram_range = (1, 1))),        #using the ngram_range parameter 
     ('Multi NB', MultinomialNB(alpha = 0.75))         
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.75      0.93      0.83      1810
         1.0       0.38      0.31      0.34      1368
         2.0       0.23      0.20      0.21       892
         3.0       0.25      0.19      0.21       314
         4.0       0.80      0.87      0.83       157

    accuracy                           0.55      4541
   macro avg       0.48      0.50      0.49      4541
weighted avg       0.50      0.55      0.52      4541



In [14]:
import spacy
nlp= spacy.load('en_core_web_sm')

In [15]:
def process (text):
    doc= nlp(text)

    processed_text= []
    for token in doc:
        if token.is_stop or token.is_punct:
            continue
        processed_text.append(token.lemma_)
        return " ".join(processed_text)
    

In [16]:
df["Clean_text"]= df['text'].apply(process)

In [17]:
df.head()

,text,subject,label_num,Clean_text
0,Donald Trump just couldn t wish all Americans ...,News,0,Donald
1,House Intelligence Committee Chairman Devin Nu...,News,0,House
2,"On Friday, it was revealed that former Milwauk...",News,0,Friday
3,"On Christmas day, Donald Trump announced that ...",News,0,Christmas
4,Pope Francis used his annual Christmas Day mes...,News,0,Pope


In [ ]:
#1. create a pipeline object
from sklearn.ensemble import RandomForestClassifier
clf = Pipeline([
    ('vectorizer_n_grams', CountVectorizer(ngram_range = (1,2))),                       #using the ngram_range parameter 
    ('random_forest', (RandomForestClassifier()))         
])

#2. fit with X_train and y_train
clf.fit(X_train, y_train)


#3. get the predictions for X_test and store it in y_pred
y_pred = clf.predict(X_test)


#4. print the classfication report
print(classification_report(y_test, y_pred))